<a href="https://colab.research.google.com/github/YehezkielFienathan/Skripsi/blob/main/Code_Analisis_Threshold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

install & import

In [ ]:
!pip install datasets sentence-transformers scikit-learn numpy pandas tqdm groq

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from tqdm import tqdm
from collections import deque, Counter
from groq import Groq
import random
import time
import re

setup client & dataset

In [ ]:
client = Groq(api_key="API_Groq")

full_dataset = load_dataset("squad", split="validation")

random.seed(42)
sample_indices = random.sample(range(len(full_dataset)), 50)
dataset = full_dataset.select(sample_indices)

print(f"Total dataset asli: {len(full_dataset)}")
print(f"Sampel yang digunakan: {len(dataset)}")

Embedding model

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/multi-qa-mpnet-base-dot-v1")

def embed_texts(texts, normalize=True):
    """
    Encode texts dengan normalisasi eksplisit.

    Args:
        texts: List of strings atau single string
        normalize: Boolean, apakah normalize embeddings

    Returns:
        numpy array of embeddings (normalized jika normalize=True)
    """
    # Pastikan input adalah list
    if isinstance(texts, str):
        texts = [texts]

    # Encode dengan normalize_embeddings parameter
    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=normalize,  # ← KEY FIX: Explicit normalization
        show_progress_bar=False,
        convert_to_numpy=True
    )

    return embeddings

print("\n=== VERIFICATION: Embedding Normalization ===")
test_texts = ["What is machine learning?", "How does AI work?"]
test_embeddings = embed_texts(test_texts, normalize=True)

# Check norms (should be ~1.0)
norms = np.linalg.norm(test_embeddings, axis=1)
print(f"Embedding norms: {norms}")
print(f"All normalized? {np.allclose(norms, 1.0)}")
print(f"Embedding shape: {test_embeddings.shape}")

In [ ]:
Retrieval

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_contexts(question_embedding, context_embeddings, contexts, threshold, top_k=5):
    """
    Retrieve contexts using COSINE SIMILARITY.

    Args:
        question_embedding: Query embedding (1D array, shape: [768])
        context_embeddings: Context embeddings (2D array, shape: [N, 768])
        contexts: List of context strings (length N)
        threshold: Minimum similarity score (float, range [-1, 1] atau [0, 1])
        top_k: Maximum number of contexts to retrieve

    Returns:
        retrieved_contexts: List of retrieved context strings
        retrieved_scores: List of cosine similarity scores
        all_similarities: All cosine similarity scores (untuk analisis)
    """
    # Reshape question_embedding untuk sklearn
    # sklearn expects shape (1, 768) untuk query
    question_emb_reshaped = question_embedding.reshape(1, -1)

    # Compute COSINE SIMILARITY
    # Output shape: (N,) setelah flatten
    similarities = cosine_similarity(context_embeddings, question_emb_reshaped).flatten()

    # Filter by threshold
    indices = np.where(similarities >= threshold)[0]

    # Jika tidak ada yang di atas threshold, return empty
    if len(indices) == 0:
        return [], [], similarities

    # Sort by similarity (descending)
    sorted_indices = indices[np.argsort(similarities[indices])[::-1]]

    # Get top_k
    top_indices = sorted_indices[:top_k]

    # Get contexts and scores
    retrieved_contexts = [contexts[i] for i in top_indices]
    retrieved_scores = [float(similarities[i]) for i in top_indices]

    return retrieved_contexts, retrieved_scores, similarities

In [ ]:
Generation dengan groq

In [ ]:
request_timestamps = deque()
answer_cache = {}

def wait_for_rate_limit():
    """Handle Groq API rate limit (30 requests/min)"""
    now = time.time()

    # Remove timestamps older than 60s
    while request_timestamps and now - request_timestamps[0] > 60:
        request_timestamps.popleft()

    # If approaching limit, wait
    if len(request_timestamps) >= 25:  # Safety margin
        wait = 60 - (now - request_timestamps[0]) + 2
        print(f"⏱️ Rate limit buffer: waiting {wait:.1f}s...")
        time.sleep(wait)
        request_timestamps.clear()

    request_timestamps.append(time.time())

def generate_answer(question, contexts):
    """
    Generate answer using Groq LLM.

    Args:
        question: Question string
        contexts: List of retrieved context strings

    Returns:
        Generated answer string
    """
    # Cache check
    cache_key = question.strip()
    if cache_key in answer_cache:
        return answer_cache[cache_key]

    # No contexts = no answer
    if len(contexts) == 0:
        return ""

    # Create prompt
    context_text = "\n\n".join(contexts)
    prompt = f"""Answer the question using ONLY the context below.
Be concise. Return only the answer, no explanation.

Context:
{context_text}

Question: {question}
Answer:"""

    # Retry logic
    for attempt in range(3):
        try:
            wait_for_rate_limit()

            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                temperature=0
            )

            answer = response.choices[0].message.content.strip()
            answer_cache[cache_key] = answer
            return answer

        except Exception as e:
            wait = 20 * (attempt + 1)
            print(f"❌ Attempt {attempt+1} failed. Waiting {wait}s... Error: {str(e)[:100]}")
            time.sleep(wait)

    return ""

Fungsi normalisasi

In [ ]:
def normalize_answer(s):
    """Normalize answer untuk fair comparison"""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def compute_exact(a_gold, a_pred):
    """Exact Match score"""
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def compute_f1(a_gold, a_pred):
    """Token-level F1 score"""
    gold_tokens = normalize_answer(a_gold).split()
    pred_tokens = normalize_answer(a_pred).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    gold_count = Counter(gold_tokens)
    pred_count = Counter(pred_tokens)
    common = sum((gold_count & pred_count).values())

    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def compute_mrr(true_context, retrieved_contexts):
    """Mean Reciprocal Rank"""
    for rank, ctx in enumerate(retrieved_contexts, start=1):
        if ctx.strip() == true_context.strip():
            return 1.0 / rank
    return 0.0

Persiapan data & cek distribusi

In [ ]:
contexts = [data["context"] for data in dataset]
answers_all = [data["answers"]["text"] for data in dataset]
questions = [data["question"] for data in dataset]

print(f"\n📊 Data Summary:")
print(f"Total questions: {len(questions)}")
print(f"Total unique contexts: {len(set(contexts))}")
print(f"Average context length: {np.mean([len(c) for c in contexts]):.0f} chars")

print("\n🔄 Embedding contexts with normalization...")
context_embeddings = embed_texts(contexts, normalize=True)

# Verify shape
print(f"Context embeddings shape: {context_embeddings.shape}")
print(f"Expected: ({len(contexts)}, 768)")

print("\n📈 Similarity Distribution Analysis:")
  sample_q_emb = embed_texts([questions[0]], normalize=True)[0]
sample_similarities = np.dot(context_embeddings, sample_q_emb)

print(f"Question: {questions[0][:80]}...")
print(f"Max similarity: {np.max(sample_similarities):.4f}")
print(f"Mean similarity: {np.mean(sample_similarities):.4f}")
print(f"Min similarity: {np.min(sample_similarities):.4f}")
print(f"Top 5 similarities: {sorted(sample_similarities, reverse=True)[:5]}")
print(f"Similarity range: [{sample_similarities.min():.4f}, {sample_similarities.max():.4f}]")

Tahap Retrieval

In [ ]:
thresholds = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # Added 0.0 as baseline

retrieval_results = []
retrieved_per_sample = {}

for threshold in thresholds:
    print(f"\n{'='*50}")
    print(f"🔍 Running Retrieval - Threshold: {threshold}")
    print(f"{'='*50}")

    recall_scores = []
    precision_scores = []
    similarity_scores = []
    mrr_scores = []
    num_retrieved_list = []
    retrieved_per_sample[threshold] = []

    for i in tqdm(range(len(questions)), desc=f"Threshold {threshold}"):
        question = questions[i]
        true_context = dataset[i]["context"]

        # Embed question (normalized)
        question_embedding = embed_texts([question], normalize=True)[0]

        # Retrieve contexts
        retrieved_contexts, retrieved_scores, all_similarities = retrieve_contexts(
            question_embedding,
            context_embeddings,
            contexts,
            threshold,
            top_k=5
        )

        # Store for generation phase
        retrieved_per_sample[threshold].append(retrieved_contexts)

        # Compute metrics
        num_retrieved_list.append(len(retrieved_contexts))
        similarity_scores.append(np.mean(retrieved_scores) if retrieved_scores else 0)
        recall_scores.append(1 if true_context in retrieved_contexts else 0)

        # Precision = relevant / retrieved (assuming top-k=5)
        if len(retrieved_contexts) > 0:
            precision_scores.append(1/len(retrieved_contexts) if true_context in retrieved_contexts else 0)
        else:
            precision_scores.append(0)

        mrr_scores.append(compute_mrr(true_context, retrieved_contexts))

    # Aggregate results
    retrieval_results.append({
        "Threshold": threshold,
        "Avg Retrieved": np.mean(num_retrieved_list),
        "Avg Similarity": np.mean(similarity_scores),
        "Recall@5": np.mean(recall_scores),
        "Precision@5": np.mean(precision_scores),
        "MRR": np.mean(mrr_scores)
    })

    print(f"\n📊 Results for threshold {threshold}:")
    print(f"  Avg contexts retrieved: {np.mean(num_retrieved_list):.2f}")
    print(f"  Avg similarity: {np.mean(similarity_scores):.4f}")
    print(f"  Recall@5: {np.mean(recall_scores):.4f}")
    print(f"  MRR: {np.mean(mrr_scores):.4f}")

print("\n" + "="*50)
print("📋 RETRIEVAL RESULTS SUMMARY")
print("="*50)
retrieval_df = pd.DataFrame(retrieval_results)
print(retrieval_df.to_string(index=False))

Tahap Generation

In [ ]:
generation_results = []

for threshold in thresholds:
    print(f"\n{'='*50}")
    print(f"💬 Running Generation - Threshold: {threshold}")
    print(f"{'='*50}")

    # Clear cache for each threshold
    answer_cache.clear()

    em_scores = []
    f1_scores = []

    for i in tqdm(range(len(questions)), desc=f"Generation {threshold}"):
        question = questions[i]
        true_answers = answers_all[i]
        retrieved_contexts = retrieved_per_sample[threshold][i]

        # Generate answer
        predicted = generate_answer(question, retrieved_contexts)

        # Show first 3 examples
        if i < 3:
            print(f"\n--- Example {i+1} ---")
            print(f"Q: {question}")
            print(f"True: {true_answers}")
            print(f"Pred: {predicted}")
            print(f"Contexts retrieved: {len(retrieved_contexts)}")

        # Compute metrics (take max across multiple gold answers)
        em = max(compute_exact(a, predicted) for a in true_answers) if true_answers else 0
        f1 = max(compute_f1(a, predicted) for a in true_answers) if true_answers else 0

        em_scores.append(em)
        f1_scores.append(f1)

        # Sleep to avoid rate limit
        time.sleep(1)

    # Aggregate results
    generation_results.append({
        "Threshold": threshold,
        "EM": np.mean(em_scores),
        "F1": np.mean(f1_scores)
    })

    print(f"\n📊 Results for threshold {threshold}:")
    print(f"  EM: {np.mean(em_scores):.4f}")
    print(f"  F1: {np.mean(f1_scores):.4f}")

print("\n" + "="*50)
print("📋 GENERATION RESULTS SUMMARY")
print("="*50)
generation_df = pd.DataFrame(generation_results)
print(generation_df.to_string(index=False))


Hasil

In [ ]:
combined_df = retrieval_df.merge(generation_df, on="Threshold")

print("\n" + "="*50)
print("📊 COMBINED RESULTS")
print("="*50)
print(combined_df.to_string(index=False))

# Find best threshold
best_f1_idx = combined_df['F1'].idxmax()
best_threshold = combined_df.loc[best_f1_idx, 'Threshold']

print(f"\n🏆 Best Threshold: {best_threshold}")
print(f"   F1 Score: {combined_df.loc[best_f1_idx, 'F1']:.4f}")
print(f"   EM Score: {combined_df.loc[best_f1_idx, 'EM']:.4f}")
print(f"   Recall@5: {combined_df.loc[best_f1_idx, 'Recall@5']:.4f}")